In [ ]:
import pandas as pd
import numpy as np

CSV_FILE_NAME = "retell_calls_mindflow.csv"

print("Iniciando Ingestão e Processamento (Fuso: Brasília | Target: > 60s)...")

# ==========================================
# FASE 1: INGESTÃO E CORREÇÃO TEMPORAL
# ==========================================

tipos = {'to_number': str, 'from_number': str, 'call_id': str}
df_original = pd.read_csv(CSV_FILE_NAME, dtype=tipos, low_memory=False)

# 1. TIMEZONE FIX: Converte para UTC e depois para Horário de Brasília (UTC-3)
df_original['created_at'] = pd.to_datetime(df_original['created_at'], utc=True)
df_original['created_at'] = df_original['created_at'].dt.tz_convert('America/Sao_Paulo')

# 2. TARGET FIX: Criando a variável alvo (Target)
# 60.000 ms = 1 minuto. Preenchemos nulos com 0 para evitar erros.
df_original['target'] = (df_original['Duracao'].fillna(0) > 60000).astype(int)

# 3. DEDUPLICAÇÃO: Remove logs duplicados
df_original = df_original.sort_values('created_at').drop_duplicates(subset=['call_id'], keep='first')

# ==========================================
# FASE 2: ENGENHARIA DE FEATURES
# ==========================================

# Ordenação Crítica por cliente e tempo
df_original = df_original.sort_values(['to_number', 'created_at']).reset_index(drop=True)
grouped = df_original.groupby('to_number')

# Extração de DDD
def get_ddd(phone):
    phone = str(phone).replace('+', '').replace('55', '')
    if not phone or len(phone) < 2: return np.nan
    return phone[:2]

df_original['ddd'] = df_original['to_number'].apply(get_ddd)

# Features Temporais (Agora em Horário de Brasília)
df_original['hora_contato'] = df_original['created_at'].dt.hour
df_original['dia_semana'] = df_original['created_at'].dt.dayofweek

# Histórico e Fadiga
df_original['n_tentativas_anteriores'] = grouped.cumcount()

df_original['primeiro_contato'] = grouped['created_at'].transform('first')
df_original['horas_desde_primeiro_contato'] = (df_original['created_at'] - df_original['primeiro_contato']).dt.total_seconds() / 3600

df_original['ultimo_contato_anterior'] = grouped['created_at'].shift(1)
df_original['horas_desde_ultimo_contato'] = (df_original['created_at'] - df_original['ultimo_contato_anterior']).dt.total_seconds() / 3600
df_original['horas_desde_ultimo_contato'] = df_original['horas_desde_ultimo_contato'].fillna(0)

# Memória Imediata e Turnos
df_original['hora_ultimo_contato'] = grouped['hora_contato'].shift(1).fillna(-1)

# Scores de Fadiga e Pressão
df_original['densidade_tentativas'] = (df_original['n_tentativas_anteriores'] / (df_original['horas_desde_primeiro_contato'] + 1)).round(4)
df_original['pressao_recente'] = (df_original['n_tentativas_anteriores'] / (df_original['horas_desde_ultimo_contato'] + 1)).round(4)

# Limpeza de Temporárias
df_original.drop(columns=['primeiro_contato', 'ultimo_contato_anterior'], inplace=True, errors='ignore')

print(f"✅ Processamento Concluído! Sucessos (Target=1): {df_original['target'].sum()}")

Iniciando Ingestão e Processamento (Fuso: Brasília | Target: > 60s)...
✅ Processamento Concluído! Sucessos (Target=1): 128


In [ ]:
# ==========================================
# FASE 3: REDUÇÃO DE DIMENSIONALIDADE E LIMPEZA FINAL
# ==========================================

print("Iniciando faxina de metadados e textos...")

# Criamos uma cópia de segurança
df_ml = df_original.copy()

# Lista de colunas para remover, incluindo as solicitadas:
# n_tentativas_anteriores, horas_desde_primeiro_contato,
# horas_desde_ultimo_contato, n_tent_manha_anteriores,
# n_tent_tarde_anteriores, n_tent_noite_anteriores
cols_to_drop = [
    'id', 'created_at', 'Nome', 'Email', 'data', 'Numero', 'status',
    'call_id', 'call_type', 'agent_id', 'agent_version', 'agent_name',
    'transcript', 'recording_url', 'disconnection_reason',
    'eleven_labs_cost', 'LLM', 'LLM_cost', 'combined_cost', 'call_summary',
    'LLM_token_usage', 'from_number', 'to_number', 'Duracao', 'Marcada',
    'segmento', 'equipe',
    'n_tentativas_anteriores', 'horas_desde_primeiro_contato',
    'horas_desde_ultimo_contato', 'n_tent_manha_anteriores',
    'n_tent_tarde_anteriores', 'n_tent_noite_anteriores'
]

# Fazemos o drop seguro
df_ml = df_ml.drop(columns=cols_to_drop, errors='ignore')

print(f"✅ Faxina concluída! O Dataset encolheu para: {df_ml.shape}")

# Visualizar as primeiras linhas para garantir que só sobraram números e categorias úteis
display(df_ml.head())

Iniciando faxina de metadados e textos...
✅ Faxina concluída! O Dataset encolheu para: (45570, 7)


,target,ddd,hora_contato,dia_semana,hora_ultimo_contato,densidade_tentativas,pressao_recente
0,0,13,15,3,-1.0,0.0000,0.0000
1,0,13,15,3,15.0,0.9203,0.9203
2,0,13,12,4,15.0,0.0938,0.0942
3,0,13,17,4,12.0,0.1125,0.4732
4,0,13,17,4,17.0,0.1496,3.6814


In [ ]:
from google.colab import files

# Salvar o DataFrame em um arquivo CSV
file_name = 'df_ml_TP.csv'
df_ml.to_csv(file_name, index=False)

# Iniciar o download para o computador local
files.download(file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_ml.describe()

,target,hora_contato,dia_semana,hora_ultimo_contato,densidade_tentativas,pressao_recente
count,45570.000000,45570.000000,45570.000000,45570.000000,45565.000000,45565.000000
mean,0.002809,13.791025,2.928154,12.251152,0.754152,97.190983
std,0.052925,3.383888,1.860532,5.499797,2.128392,324.618496
min,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000
25%,0.000000,11.000000,1.000000,10.000000,0.041500,0.188400
50%,0.000000,14.000000,3.000000,13.000000,0.093800,0.995900
75%,0.000000,16.000000,5.000000,16.000000,0.280700,6.400500
max,1.000000,23.000000,6.000000,23.000000,22.932900,2131.094600


In [ ]:
# Preenchendo valores nulos com 0
df_ml = df_ml.fillna(0)

# Mapeamento de valores nulos no df_ml (verificação)
null_summary = pd.DataFrame({
    'Nulos': df_ml.isnull().sum(),
    'Percentual (%)': (df_ml.isnull().sum() / len(df_ml) * 100).round(2)
})

print("Resumo de valores nulos por coluna após preenchimento:")
display(null_summary[null_summary['Nulos'] > 0].sort_values(by='Nulos', ascending=False))

if df_ml.isnull().sum().sum() == 0:
    print("✅ Todos os valores nulos foram preenchidos com 0.")
else:
    print("⚠️ Ainda existem valores nulos no dataset.")

Resumo de valores nulos por coluna após preenchimento:


,Nulos,Percentual (%)


✅ Todos os valores nulos foram preenchidos com 0.


In [ ]:
df_ml.columns

Index(['target', 'ddd', 'hora_contato', 'dia_semana', 'hora_ultimo_contato',
       'densidade_tentativas', 'pressao_recente'],
      dtype='object')